In [1]:
import time
import torch as th
from ethos.utils import load_model_from_checkpoint

# 配置
MODEL_PATH = "/home/ligong1/mednlp/ethos/ethos-paper/out/mimic_layer_6_batch_32_do_0.3/best_model.pt"
DEVICE = "cuda" if th.cuda.is_available() else "cpu"
N_TOKENS = 1000

# 1. 加载并移动模型
model, _ = load_model_from_checkpoint(MODEL_PATH, DEVICE, for_training=False)
model.to(DEVICE) 
model.eval()



number of parameters: 45.87M


Ethos(
  (transformer): ModuleDict(
    (wte): Embedding(4416, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.3, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=False)
          (c_proj): Linear(in_features=768, out_features=768, bias=False)
          (attn_dropout): Dropout(p=0.3, inplace=False)
          (resid_dropout): Dropout(p=0.3, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=False)
          (dropout): Dropout(p=0.3, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=768, out_features=4416, bias=False)
)

In [5]:
def benchmark(use_cache, top_k=None):
    # 初始化：1个病人，100个初始Token
    curr_tokens = th.randint(0, model.config.vocab_size, (1, 2024), device=DEVICE)
    kv_caches = None
    last_token = None
    
    th.cuda.synchronize()
    start_time = time.time()
    
    for _ in range(N_TOKENS):
        # 参考 run_inference.py 逻辑：
        # 1. 第一次或无缓存模式，传入全部 tokens
        # 2. 有缓存模式，仅传入上一个产生的 last_token
        input_to_model = curr_tokens if (kv_caches is None or not use_cache) else last_token
        
        last_token, new_cache = model.get_next_token(
            input_to_model,
            top_k=top_k, 
            kv_caches=kv_caches if use_cache else None
        )
        
        if use_cache:
            kv_caches = new_cache
            
        curr_tokens = th.cat([curr_tokens, last_token], dim=1)
        
        # 模拟滑动窗口：超过 block_size 时重置缓存（参考 run_inference.py 的 offset 逻辑）
        if curr_tokens.size(1) >= model.config.block_size:
            curr_tokens = curr_tokens[:, -model.config.block_size:]
            kv_caches = None
            use_cache = None

    th.cuda.synchronize()
    return time.time() - start_time

print(f"设备: {DEVICE} | 生成数量: {N_TOKENS}")



设备: cuda | 生成数量: 1000


In [3]:
# 测试无缓存速度
t_orig = benchmark(use_cache=False)
print(f"原始模式 (No Cache): {t_orig:.2f}s | 速度: {N_TOKENS/t_orig:.1f} tok/s")



原始模式 (No Cache): 27.38s | 速度: 36.5 tok/s


In [6]:
# 测试有缓存速度
t_opt = benchmark(use_cache=True)
print(f"优化模式 (With Cache): {t_opt:.2f}s | 速度: {N_TOKENS/t_opt:.1f} tok/s")



优化模式 (With Cache): 26.59s | 速度: 37.6 tok/s


In [22]:
print(f"提升倍数: {t_orig/t_opt:.2f}x")

提升倍数: 6.77x


In [12]:
# 测试token裁剪速度
t_crop = benchmark(use_cache=False, top_k=5000)
print(f"token裁剪模式 (No Cache, top_k=5000): {t_crop:.2f}s | 速度: {N_TOKENS/t_crop:.1f} tok/s")



token裁剪模式 (No Cache, top_k=5000): 55.36s | 速度: 36.1 tok/s


In [24]:
# 测试缓存与裁剪
t_crop_opt = benchmark(use_cache=True, top_k=5000)
print(f"缓存与裁剪模式 (With Cache, top_k=5000): {t_crop_opt:.2f}s | 速度: {N_TOKENS/t_crop_opt:.1f} tok/s")

缓存与裁剪模式 (With Cache, top_k=5000): 3.16s | 速度: 316.8 tok/s


In [25]:
def test_n(n):
    orig_time = 0
    opt_time = 0
    crop_time = 0
    crop_opt_time = 0

    for i in range(n):
        t_orig = benchmark(use_cache=False)
        t_opt = benchmark(use_cache=True)
        t_crop = benchmark(use_cache=False, top_k=5000)
        t_crop_opt = benchmark(use_cache=True, top_k=5000)
        orig_time += t_orig
        opt_time += t_opt
        crop_time += t_crop
        crop_opt_time += t_crop_opt

        print(f"第{i}次测试: 原始={t_orig:.2f}s, 优化={t_opt:.2f}s, 裁剪={t_crop:.2f}s, 缓存裁剪={t_crop_opt:.2f}s")
    
    print(f"30次测试平均: 原始={orig_time/30:.2f}s\n优化={opt_time/30:.2f}s\n裁剪={crop_time/30:.2f}s\n缓存裁剪={crop_opt_time/30:.2f}s")

test_n(30)

第0次测试: 原始=19.91s, 优化=2.88s, 裁剪=20.51s, 缓存裁剪=3.13s
第1次测试: 原始=19.99s, 优化=2.85s, 裁剪=20.51s, 缓存裁剪=3.20s
第2次测试: 原始=19.97s, 优化=2.85s, 裁剪=20.52s, 缓存裁剪=3.11s
第3次测试: 原始=20.07s, 优化=2.85s, 裁剪=20.60s, 缓存裁剪=3.13s
第4次测试: 原始=20.14s, 优化=2.87s, 裁剪=20.60s, 缓存裁剪=3.12s
第5次测试: 原始=20.11s, 优化=2.84s, 裁剪=20.58s, 缓存裁剪=3.19s
第6次测试: 原始=20.16s, 优化=2.86s, 裁剪=20.72s, 缓存裁剪=3.14s
第7次测试: 原始=20.21s, 优化=2.87s, 裁剪=20.82s, 缓存裁剪=3.16s
第8次测试: 原始=20.10s, 优化=2.90s, 裁剪=20.70s, 缓存裁剪=3.14s
第9次测试: 原始=20.15s, 优化=2.93s, 裁剪=20.71s, 缓存裁剪=3.19s
第10次测试: 原始=20.22s, 优化=2.89s, 裁剪=20.82s, 缓存裁剪=3.15s
第11次测试: 原始=20.21s, 优化=2.92s, 裁剪=20.73s, 缓存裁剪=3.12s
第12次测试: 原始=20.15s, 优化=2.88s, 裁剪=20.80s, 缓存裁剪=3.14s
第13次测试: 原始=20.12s, 优化=2.86s, 裁剪=20.73s, 缓存裁剪=3.17s
第14次测试: 原始=20.18s, 优化=2.87s, 裁剪=20.59s, 缓存裁剪=3.15s
第15次测试: 原始=20.10s, 优化=2.85s, 裁剪=20.60s, 缓存裁剪=3.15s
第16次测试: 原始=20.14s, 优化=2.92s, 裁剪=20.65s, 缓存裁剪=3.15s
第17次测试: 原始=20.11s, 优化=2.86s, 裁剪=20.61s, 缓存裁剪=3.14s
第18次测试: 原始=20.18s, 优化=2.87s, 裁剪=20.60s, 缓存裁剪=3.13s
第19次测试: 原始=20.08s, 优化=2.86s, 裁剪=20.81s, 缓